In [1]:
import numpy as np
from linear_regression import LinearRegression

In [2]:
raw = np.genfromtxt(
    "../Data/housing.csv",
    delimiter=",",
    skip_header=1,
    dtype=str,
    encoding="utf-8"
)

raw.shape

(20640, 10)

In [3]:
mask = np.all(np.char.strip(raw[:, 0:9]) != "", axis=1)
raw = raw[mask]

raw.shape

(20433, 10)

In [4]:
raw[raw == ""] = np.nan
X_num = raw[:, 0:8].astype(float)
y = raw[:, 8].astype(float)
x_cat = raw[:, 9]

X = np.column_stack([X_num, x_cat])
categorical_cols = [8]

X.shape, y.shape

((20433, 9), (20433,))

In [5]:
model = LinearRegression(confidence_level=0.95, add_intercept=True)
model.fit(X, y, categorical_cols=categorical_cols)

## Sammanfattning av regressionsmodellen

En multipel linjär regressionsmodell har anpassats till data med både numeriska och kategoriska variabler. Modellen inkluderar intercept och estimerades med minsta kvadratmetoden.

In [6]:
model.n

20433

In [7]:
model.d

12

In [8]:
model.sample_variance()

4713776929.49694

In [9]:
model.sample_std()

68656.95106467327

In [10]:
model.rmse()

68635.10693083824

In [11]:
model.r2()

0.6464638320299037

## Modellens förklaringsgrad (R²)

Modellens determinationskoefficient är
$$
R^2 = \frac{SSR}{S_{yy}}
$$
där $S_{yy}$ är den totala variationen i responsvariabeln och $SSR$ är den del som förklaras av regressionsmodellen.

I denna analys är $R^2 \approx 0.65$, vilket innebär att cirka 65 % av variationen i responsvariabeln förklaras av de inkluderade förklarande variablerna, medan cirka 35 % återstår som residualvariation (SSE).

Detta är en relativt hög förklaringsgrad i många samhällsvetenskapliga sammanhang, men det innebär samtidigt att modellen inte är fullständig och att ytterligare variabler eller icke-linjära samband kan finnas.

Det är viktigt att notera att R² inte är en konfidensnivå, utan ett mått på modellens förklaringskraft.

In [12]:
model.regression_significance()

{'F': 3111.607959341615, 'df1': 12, 'df2': 20420, 'p_value': 0.0}

## Global signifikans – F-test

För att testa om modellen som helhet är statistiskt signifikant används det globala F-testet:

$$\frac{SSR/d}{S^2} \sim F(d, n-d-1)$$

där

$d$ = 12 är antalet regressionsparametrar (exklusive intercept)

$n$ = 20433 är stickprovsstorleken

frihetsgraderna är alltså $F(12, 20420)$

Nollhypotesen är:

$H_0: \beta_1 = \beta_2 = \dots = \beta_d = 0$

Det innebär att ingen av de förklarande variablerna har något linjärt samband med responsen.

Det observerade F-värdet är mycket stort (≈ 3111) och p-värdet är ≈ 0, vilket innebär att nollhypotesen förkastas på 5%-nivån.

Slutsats: Modellen är statistiskt signifikant som helhet.

In [14]:
model.coef_tests()

{'beta': array([-2.26995411e+06, -2.68129892e+04, -2.54821847e+04,  1.07252004e+03,
        -6.19326372e+00,  1.00556290e+02, -3.79690829e+01,  4.96173262e+01,
         3.92595729e+04, -3.92843003e+04,  1.52901941e+05, -3.95405158e+03,
         4.27813436e+03]),
 'std_error': array([8.80138815e+04, 1.01965063e+03, 1.00470188e+03, 4.38857068e+01,
        7.91469396e-01, 6.86850715e+00, 1.07614781e+00, 7.45131075e+00,
        3.38004750e+02, 1.74425776e+03, 3.07418787e+04, 1.91333945e+03,
        1.56952541e+03]),
 't': array([-25.79086465, -26.29625121, -25.36293136,  24.43893739,
         -7.82501983,  14.64019589, -35.28240514,   6.65887222,
        116.15095003, -22.52207286,   4.97373445,  -2.06657088,
          2.7257503 ]),
 'p_value': array([2.29020185e-144, 6.59645179e-150, 9.39227574e-140, 4.85480716e-130,
        5.32160039e-015, 2.73458708e-048, 9.34949768e-265, 2.82953190e-011,
        0.00000000e+000, 5.66274508e-111, 6.62082244e-007, 3.87871844e-002,
        6.42101969e-00

## Individuell signifikans – t-test

För varje enskild regressionskoefficient används t-testet:

$$\frac{\hat{\beta}_i}{S\sqrt{c_{ii}}} \sim T(n-d-1)$$

med frihetsgrader $n-d-1 = 20420.$

Nollhypotesen är:

$$H_0: \beta_i = 0$$

Detta testar om en specifik variabel bidrar till modellen givet att övriga variabler hålls konstanta.

Flera variabler har mycket låga p-värden (< 0.05), vilket innebär att de är statistiskt signifikanta. Några variabler har dock p-värden över 0.05, vilket innebär att vi inte kan förkasta hypotesen att deras sanna koefficient är noll.

Detta indikerar att alla inkluderade parametrar kanske inte är nödvändiga i modellen.

In [15]:
model.confidence_intervals()

{'confidence_level': 0.95,
 'lower': array([-2.44246837e+06, -2.88115862e+04, -2.74514810e+04,  9.86500538e+02,
        -7.74460719e+00,  8.70934655e+01, -4.00784188e+01,  3.50121598e+01,
         3.85970565e+04, -4.27031853e+04,  9.26453945e+04, -7.70435028e+03,
         1.20173873e+03]),
 'upper': array([-2.09743984e+06, -2.48143922e+04, -2.35128885e+04,  1.15853955e+03,
        -4.64192026e+00,  1.14019115e+02, -3.58597469e+01,  6.42224926e+01,
         3.99220893e+04, -3.58654153e+04,  2.13158488e+05, -2.03752883e+02,
         7.35452999e+03]),
 't_crit': 1.9600801652187363,
 'df': 20420}

## Konfidensintervall och konfidensnivå

Ett 95% konfidensintervall för en koefficient innebär att om vi upprepar stickprovsproceduren många gånger, kommer cirka 95% av intervallen att innehålla den sanna parametern.

Valet av 95% konfidensnivå motsvarar $\alpha = 0.05$, vilket är en standardnivå i statistisk inferens.

Konfidensnivån är kopplad till signifikansnivån i hypotesprövningen, men har ingen direkt koppling till R².

Ett intervall som inte innehåller 0 motsvarar ett signifikant resultat i t-testet på 5%-nivån.

In [16]:
R, P = model.pearson_matrix(X, categorical_cols=categorical_cols)
R.shape

(12, 12)

## Multikollinearitet

Pearsons korrelationsmatris visar att vissa förklarande variabler har mycket hög inbördes korrelation (upp till ≈ 0.98).

En så hög korrelation indikerar stark multikollinearitet, vilket kan:

öka standardfelen

göra koefficienterna instabila

försvåra tolkningen av individuella effekter

Multikollinearitet påverkar dock inte modellens totala förklaringsgrad (R²) eller det globala F-testet direkt, men kan göra det svårt att avgöra vilken av de starkt korrelerade variablerna som faktiskt driver sambandet.

In [18]:
absR = np.abs(R)
np.fill_diagonal(absR, 0)
i, j = np.unravel_index(np.argmax(absR), absR.shape)
i, j, R[i,j], P[i,j]

(np.int64(4), np.int64(6), np.float64(0.9797282708045647), np.float64(0.0))

In [17]:
R[:10, :10]

array([[ 1.        , -0.92461611, -0.10935655,  0.04548017,  0.06960802,
         0.1002703 ,  0.05651277, -0.01555015, -0.05533745,  0.00950071],
       [-0.92461611,  1.        ,  0.01189907, -0.03666681, -0.06698283,
        -0.10899734, -0.07177419, -0.07962632,  0.35108357, -0.01666228],
       [-0.10935655,  0.01189907,  1.        , -0.3606283 , -0.32045104,
        -0.2957873 , -0.30276797, -0.11827772, -0.23696771,  0.01710531],
       [ 0.04548017, -0.03666681, -0.3606283 ,  1.        ,  0.9303795 ,
         0.85728125,  0.91899153,  0.19788152,  0.0264775 , -0.00760262],
       [ 0.06960802, -0.06698283, -0.32045104,  0.9303795 ,  1.        ,
         0.87774674,  0.97972827, -0.00772285, -0.00646289, -0.00436147],
       [ 0.1002703 , -0.10899734, -0.2957873 ,  0.85728125,  0.87774674,
         1.        ,  0.9071859 ,  0.00508662, -0.01960181, -0.01045053],
       [ 0.05651277, -0.07177419, -0.30276797,  0.91899153,  0.97972827,
         0.9071859 ,  1.        ,  0.01343389

## Sammanfattande bedömning

Modellen är statistiskt signifikant som helhet och förklarar en betydande del av variationen i responsvariabeln (R² ≈ 0.65).

Samtidigt visar analysen att:

**vissa variabler inte är individuellt signifikanta**

**stark multikollinearitet förekommer**

En möjlig förbättring vore att estimera en reducerad modell eller hantera multikollinearitet, exempelvis genom variabelselektion eller dimensionreduktion.